# Cruzamento de Dados - Telefones de Alunos
### Objetivo: Cruzar os dados de email dos Check-ins das Aulas Semanais para buscar o último telefone válido dos leads.

In [1]:
# Importações básicas
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')

#PRODUCT Ids
CRONOGRAMA_ID = '3310674'

#Subdomains do Hotmart Club
CRONOGRAMA_SUBDOMAIN = 'cronogramadosfluentes-xwamel'

#Aulas de Conversação 
AULAS_DE_CONVERSACAO_SUBDOMAIN = 'aulasdeconversacao-vpqdjd'

#Ingles Express Travel
INGLES_EXPRESS_TRAVEL_SUBDOMAIN = 'inglesexpresstravel'

#Intensivão do Inglês
INTENSIVAO_DO_INGLES_BASICO_SUBDOMAIN = 'intensivaodoinglesbasico'

#Aulas de Conversação Particulares
AULAS_DE_CONVERSACAO_PARTICULARES_SUBDOMAIN = 'aulasdeconversacao-avjbpp'

# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df, 
    timestamp_ms_to_datetime
)

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc, load_query_from_file

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart

# utilitários tmb
from tmb_utils import TMB   

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()

# Inicializar API TMB
tmb = TMB()

print('✓ Importações concluídas com sucesso!')

✓ Importações concluídas com sucesso!


# Buscando Check-ins na Aula Definida


In [2]:
df = load_raw_data('elegible_students.csv')

df['email'] = clean_and_lower_column(df, 'email')

df.shape

(13108, 2)

## Buscando Emails e Phones para Cruzamento

In [3]:
df_emails = db.execute_query_from_file('select_all_emails.sql')

df_emails['email'] = clean_and_lower_column(df_emails, 'email')

print('df_emails.shape', df_emails.shape)

df_phones = db.execute_query_from_file('select_leads_last_phone.sql')

print('df_phones.shape', df_phones.shape)

df_names = db.execute_query_from_file('select_all_names.sql')

print('df_names.shape', df_names.shape)

df_emails.shape (318808, 6)
df_phones.shape (393625, 11)
df_names.shape (375650, 2)


### Cruzando E-mails com df_checkin para validar lead_id

In [4]:
df = pd.merge(df, df_emails, on='email', how='left')

### Cruzando checkins com Phones

In [5]:
df = pd.merge(df, df_phones, on='lead_id', how='left')

# Cruzando Checkins com Names

In [6]:
df = pd.merge(df, df_names, on='lead_id', how='left')

## Limpeza de Dados


In [7]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(df_checkins_phones):
    # Drop columns: 'session_id', 'user_id' and 15 other columns
    df_checkins_phones = df_checkins_phones.drop(columns=['session_id', 'user_id', 'expiration_date', 'class_date', 'is_active', 'id_x', 'active_id', 'created_at_y', 'checked', 'id_y', 'raw_phone_input', 'phone_type', 'ddd', 'ddi', 'is_valid', 'created_at', 'reprocessed'])
    # Rename column 'created_at_x' to 'check_in_date'
    df_checkins_phones = df_checkins_phones.rename(columns={'created_at_x': 'check_in_date'})
    # Drop rows with missing data in column: 'formatted_phone'
    df_checkins_phones = df_checkins_phones.dropna(subset=['formatted_phone'])
    # Drop duplicate rows in column: 'lead_id'
    df_checkins_phones = df_checkins_phones.drop_duplicates(subset=['lead_id'])
    # Drop duplicate rows in column: 'formatted_phone'
    df_checkins_phones = df_checkins_phones.drop_duplicates(subset=['formatted_phone'])
    return df_checkins_phones

df_checkins_names_clean = clean_data(df_checkins_names.copy())
df_checkins_names_clean.head()

,check_in_date,utm_source,utm_medium,utm_campaign,utm_term,utm_content,session_type,email,lead_id,formatted_phone,whatsapp_format,name
0,2025-12-02 03:21:51.156718+00:00,whatsapp,grupos,aulas_semanais,None,msg3,live,simone.soaresfernandes@gmail.com,400db87e-be00-49d0-b428-b35a6adc5e92,+556183421310,+556183421310,SImone Fernandes
1,2025-12-02 14:35:17.838198+00:00,whatsapp,grupos,isca-aulassemanais,None,0212_msg_1,live,vemaiaa23@gmail.com,d9ec31b6-b84b-4579-9c60-4ad1df688351,+5591985377560,+559185377560,Vitória Emanuelly
2,2025-12-02 14:49:29.297243+00:00,whatsapp,grupos,isca-aulassemanais,None,0212_msg_1,live,elenaurbanavicius@yahoo.com.br,e00796d2-31bd-4bd4-8f21-57efb3828619,+554196031531,+554196031531,Elena
3,2025-12-02 15:07:17.057916+00:00,whatsapp,grupos,isca-aulassemanais,None,0212_msg_1,live,ninaxav@hotmail.com,e0551e75-bea6-43ba-a9f0-975b3195e9ea,+351966341295,+351966341295,M Monte Xavier
4,2025-12-02 15:07:22.382587+00:00,whatsapp,grupos,isca-aulassemanais,None,0212_msg_1,live,darlana_paulino@hotmail.com,1bc7a9c6-1167-4c21-8873-c2532d8f9580,+554699703292,+554699703292,Darlana


## Buyers

In [8]:
df_buyers = db.execute_query_from_file('select_generic_sales', params={'product_name': 'Cronograma dos Fluentes'})

df_buyers['phone'] = process_phone_string(df_buyers, 'phone')

print('Df Shape buyers:', df_buyers.shape)

Df Shape buyers: (38098, 9)


In [9]:
df_hotmart = load_raw_data('hotmart.csv', sep=';')

def clean_data(hotmart_transactions):
    # Drop columns: 'Nome do Produtor', 'Documento do Produtor' and 49 other columns
    hotmart_transactions = hotmart_transactions.drop(columns=['Nome do Produtor', 'Documento do Produtor', 'Nome do Afiliado', 'Transação', 'Meio de Pagamento', 'Origem', 'Moeda', 'Preço do Produto', 'Moeda.1', 'Preço da Oferta', 'Taxa de Câmbio', 'Moeda.2', 'Preço Original', 'Número da Parcela', 'Data de Venda', 'Data de Confirmação', 'Status', 'Documento', 'CEP', 'Cidade', 'Estado', 'Bairro', 'País', 'Endereço', 'Número', 'Complemento', 'chave', 'Código do Produto', 'Código da Afiliação', 'Tipo de Pagamento', 'Período Grátis', 'Tem co-produção', 'Venda feita como', 'Preço Total', 'Tipo pagamento oferta', 'Taxa de Câmbio Real', 'Preço Total Convertido', 'Quantidade de itens', 'Oferta de Upgrade', 'Cupom', 'Moeda.3', 'Valor que você recebeu convertido', 'Data Vencimento', 'Taxa de Câmbio do valor recebido', 'Instagram', 'Origem da venda', 'Moeda de recebimento', 'Faturamento líquido', 'Código do assinante', 'Nota Fiscal', 'Valor do frete bruto'])
    # Filter rows based on column: 'Recorrência'
    hotmart_transactions = hotmart_transactions[(hotmart_transactions['Recorrência'] == 1) | (hotmart_transactions['Recorrência'].isna())]
    hotmart_transactions['DDD'] = process_phone_string(hotmart_transactions, 'DDD')
    hotmart_transactions['Telefone'] = process_phone_string(hotmart_transactions, 'Telefone')
    hotmart_transactions['phone'] = hotmart_transactions.apply(lambda x: x['Telefone'] if pd.isna(x['DDD']) else f'55{x["DDD"]}{x["Telefone"]}', axis=1)
    # Drop column: 'DDD'
    hotmart_transactions = hotmart_transactions.drop(columns=['DDD', 'Telefone'])

    # Drop column: 'Recorrência'
    hotmart_transactions = hotmart_transactions.drop(columns=['Recorrência'])
    # Rename column 'Nome do Produto' to 'product_name'
    hotmart_transactions = hotmart_transactions.rename(columns={'Nome do Produto': 'product_name', "Nome": "name", "Email": "email","Código de Oferta": "offer_id" })
    # Drop column: 'Origem de Checkout'
    hotmart_transactions = hotmart_transactions.drop(columns=['Origem de Checkout'])
    hotmart_transactions = hotmart_transactions[hotmart_transactions['product_name'] == 'Cronograma dos Fluentes ']
    return hotmart_transactions

df_hotmart = clean_data(df_hotmart.copy())


df_tmb = load_raw_data('tmb.csv', sep=';')

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_tmb):
    # Drop columns: 'Produtor', 'Pedido' and 29 other columns
    df_tmb = df_tmb.drop(columns=['Produtor', 'Pedido', 'Endereço completo', 'Logradouro', 'Número', 'Bairro', 'Complemento', 'CEP', 'Estado', 'Cidade', 'País', 'Status', 'Status Financeiro', 'Status Cancelamento', 'Ticket (R$)', 'Modalidade de Contrato', 'Criado Em', 'Data Efetivado', 'Data Cancelado', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_content', 'utm_last_source', 'utm_last_medium', 'utm_last_campaign', 'utm_last_content', 'Renovação automatica', 'Data de Renovação Prevista', 'Renovação enviada', 'Oferta'])
    # Drop column: 'Cliente CPF'
    df_tmb = df_tmb.drop(columns=['Cliente CPF'])
    # Rename column 'Produto' to 'offer_code'
    df_tmb = df_tmb.rename(columns={'Produto': 'offer_id', "Cliente Nome": "name", "Cliente Email": "email", "Telefone": "phone"})

    df_tmb['phone'] = process_phone_string(df_tmb, 'phone')
    
    return df_tmb

df_tmb = clean_data(df_tmb.copy())

In [10]:
df_hotmart_pt_1 = flatten_list_to_df(hotmart.get_sales_history(
    start_date='2023-11-01',
    end_date='2024-12-31',
))

df_hotmart_pt_2 = flatten_list_to_df(hotmart.get_sales_history(
    start_date='2025-01-01',
    end_date='2025-12-31',
))

df_hotmart_2 = pd.concat([df_hotmart_pt_1, df_hotmart_pt_2])

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_hotmart):
    # Filter rows based on column: 'purchase_recurrency_number'
    df_hotmart = df_hotmart[(df_hotmart['purchase_recurrency_number'] == 1) | (df_hotmart['purchase_recurrency_number'].isna())]
    # Drop columns: 'purchase_hotmart_fee_currency_code', 'purchase_hotmart_fee_total' and 22 other columns
    df_hotmart = df_hotmart.drop(columns=['purchase_hotmart_fee_currency_code', 'purchase_hotmart_fee_total', 'purchase_hotmart_fee_fixed', 'purchase_hotmart_fee_percentage', 'purchase_hotmart_fee_base', 'purchase_is_subscription', 'purchase_offer_payment_mode', 'purchase_tracking_source', 'purchase_payment_type', 'purchase_payment_installments_number', 'purchase_commission_as', 'purchase_warranty_expire_date', 'purchase_order_date', 'purchase_price_value', 'purchase_price_currency_code', 'producer_ucode', 'product_id', 'producer_name', 'buyer_ucode', 'purchase_tracking_source_sck', 'purchase_recurrency_number', 'purchase_tracking_external_code', 'purchase_payment_method', 'purchase_status'])
    # Rename column 'purchase_transaction' to 'transaction'
    df_hotmart = df_hotmart.rename(columns={'purchase_transaction': 'transaction'})
    # Rename column 'purchase_offer_code' to 'offer_code'
    df_hotmart = df_hotmart.rename(columns={'purchase_offer_code': 'offer_code'})
    # Rename column 'purchase_approved_date' to 'approved_at'
    df_hotmart = df_hotmart.rename(columns={'purchase_approved_date': 'approved_at'})
    # Rename column 'buyer_name' to 'name'
    df_hotmart = df_hotmart.rename(columns={'buyer_name': 'name'})
    # Rename column 'buyer_email' to 'email'
    df_hotmart = df_hotmart.rename(columns={'buyer_email': 'email'})
    # Rename column 'approved_at' to 'approval_date'
    df_hotmart = df_hotmart.rename(columns={'approved_at': 'approval_date'})
    df_hotmart['approval_date'] = timestamp_ms_to_datetime(df_hotmart, 'approval_date')
    return df_hotmart

df_hotmart_2 = clean_data(df_hotmart_2.copy())
df_hotmart_2.head()

✓ Concluído: 387 página(s) processada(s). Total de 19335 itens obtidos.)
✓ Concluído: 349 página(s) processada(s). Total de 17416 itens obtidos.)


,email,name,product_name,offer_code,approval_date,transaction
0,oseildopedrosa@hotmail.com,Oseildo Pedrosa Josiele Noronha,Cronograma dos Fluentes,zyvtnv0d,2023-11-01 13:48:19,HP3359038232
1,oseildopedrosa@hotmail.com,Oseildo Pedrosa Josiele Noronha,Manual Prático de Conversação em Inglês,g1d6dz6j,2023-11-01 13:48:22,HP3050854363
2,georgiamattei@hotmail.com,Geórgia Mattei,Cronograma dos Fluentes,zyvtnv0d,2023-11-01 15:08:41,HP1200458985
3,osmarina.pessoa@gmail.com,Osmarina dos Anjos Pessoa,Cronograma dos Fluentes,zyvtnv0d,2023-11-01 16:01:33,HP3570912282
4,juliano_rdf@hotmail.com,Juliano Rodrigues de Freitas,Cronograma dos Fluentes,zyvtnv0d,2023-11-01 16:02:48,HP3632472106


In [11]:
df_tmb_2 = flatten_list_to_df(tmb.get_pedidos(page_size=1000))

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_tmb):
    # Filter rows based on column: 'status_pedido'
    df_tmb = df_tmb[df_tmb['status_pedido'] == "Efetivado"]
    # Drop columns: 'produtor', 'status_pedido' and 29 other columns
    df_tmb = df_tmb.drop(columns=['produtor', 'status_pedido', 'documento', 'valor_total', 'parcelas', 'valor_entrada', 'valor_parcela', 'taxa_administracao', 'endereco_completo', 'criado_em', 'telefone', 'produto_id', 'nascimento', 'produtor_id', 'utm_source', 'utm_medium', 'utm_content', 'utm_campaign', 'utm_last_source', 'melhor_dia_pagamento', 'utm_last_medium', 'utm_last_content', 'utm_last_campaign', 'pais', 'cep', 'endereco_estado', 'endereco_cidade', 'endereco_bairro', 'endereco_logradouro', 'endereco_numero', 'endereco_complemento'])
    # Rename column 'lancamento' to 'offer_code'
    df_tmb = df_tmb.rename(columns={'lancamento': 'offer_code'})
    # Rename column 'pedido_id' to 'transaction'
    df_tmb = df_tmb.rename(columns={'pedido_id': 'transaction'})
    # Rename column 'cliente' to 'name'
    df_tmb = df_tmb.rename(columns={'cliente': 'name'})
    # Rename column 'data_efetivado' to 'approval_date'
    df_tmb = df_tmb.rename(columns={'data_efetivado': 'approval_date'})
    # Change column type to datetime64[ns] for column: 'approval_date'
    df_tmb = df_tmb.astype({'approval_date': 'datetime64[ns]'})
    return df_tmb

df_tmb_2 = clean_data(df_tmb_2.copy())
df_tmb_2.head()

✓ Concluído: 1 página(s) processada(s). Total de 100 pedidos obtidos.


,offer_code,transaction,name,email,approval_date,status_financeiro
0,COM-BF25-alunos-Oferta Negociação,1918767,Matheus Ruas,matheusruas@hotmail.com,2025-12-02 18:53:01.355239,Adimplente
1,COM-BF25-alunos-Oferta Negociação,1917023,KLEBER FERREIRA DO VALE,klebervaleadvogado@outlook.com,2025-12-02 19:00:26.471830,Adimplente
2,COM-BF25-alunos-Oferta Negociação,1915079,Kyara Costa de Azevedo,kyky338.157@gmail.com,2025-12-01 10:55:35.356717,Adimplente
3,COM-BF25-Oferta Downsell-3,1914968,Bruno Luis Reis Intaschi,brunointaschi@hotmail.com,2025-12-01 11:21:29.853887,Adimplente
4,COM-BF25-Oferta Downsell-3,1914804,Joana Darc Santos Almeida Oliveira,oliveiragiseli071@gmail.com,2025-12-01 09:30:20.814242,Adimplente


In [12]:
# Lead Ids Set
lead_ids_to_exclude = set()
lead_ids_to_exclude = lead_ids_to_exclude.union(set(df_buyers['lead_id'].unique()))

# Emails Set
emails_to_exclude = set()
emails_to_exclude = emails_to_exclude.union(set(df_buyers['email'].unique()))
emails_to_exclude = emails_to_exclude.union(set(df_tmb['email'].unique()))
emails_to_exclude = emails_to_exclude.union(set(df_hotmart['email'].unique()))
emails_to_exclude = emails_to_exclude.union(set(df_hotmart_2['email'].unique()))
emails_to_exclude = emails_to_exclude.union(set(df_tmb_2['email'].unique()))

# Phone Set
phones_to_exclude = set()
phones_to_exclude = phones_to_exclude.union(set(df_buyers['phone'].unique()))
phones_to_exclude = phones_to_exclude.union(set(df_tmb['phone'].unique()))
phones_to_exclude = phones_to_exclude.union(set(df_hotmart['phone'].unique()))

In [13]:
df_checkins_names_clean['phone'] = process_phone_string(df_checkins_names_clean, 'formatted_phone')
df_checkins_names_clean = df_checkins_names_clean[~df_checkins_names_clean['lead_id'].isin(lead_ids_to_exclude)]
df_checkins_names_clean = df_checkins_names_clean[~df_checkins_names_clean['email'].isin(emails_to_exclude)]
df_checkins_names_clean = df_checkins_names_clean[~df_checkins_names_clean['phone'].isin(phones_to_exclude)]
df_checkins_names_clean.head()
df_checkins_names_clean.shape


(3815, 13)

In [12]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(df):
    # Drop column: 'transaction_history'
    df = df.drop(columns=['transaction_history'])
    # Drop columns: 'id_x', 'created_at_x' and 9 other columns
    df = df.drop(columns=['id_x', 'created_at_x', 'checked', 'id_y', 'raw_phone_input', 'ddd', 'ddi', 'is_valid', 'created_at_y', 'reprocessed', 'whatsapp_format'])
    # Drop duplicate rows in column: 'lead_id'
    df = df.drop_duplicates(subset=['lead_id'])
    # Drop duplicate rows in column: 'email'
    df = df.drop_duplicates(subset=['email'])
    # Drop duplicate rows in column: 'formatted_phone'
    df = df.drop_duplicates(subset=['formatted_phone'])
    # Drop rows with missing data in column: 'lead_id'
    df = df.dropna(subset=['lead_id'])
    # Drop rows with missing data in column: 'formatted_phone'
    df = df.dropna(subset=['formatted_phone'])
    # Rename column 'formatted_phone' to 'phone'
    df = df.rename(columns={'formatted_phone': 'phone'})
    df['phone'] = df['phone'].apply(lambda x: f'+{x.replace("+","")}')
    return df

df_clean = clean_data(df.copy())
df_clean.head()

,email,lead_id,active_id,phone,phone_type,name
0,006fabi@gmail.com,29d0f6ae-cb89-4225-8ddb-70dc9071daa9,5,+5531972471101,Nacional,Fabiola Rodrigues
2,0311cintia@gmail.com,9cbfc74c-3382-48e8-bf55-7d095484a134,None,+5521980586255,Nacional,Cintia dos Santos Nascimento
3,054996663854tbrazr@gmail.com,13890b33-78c6-4bc7-893d-9f4908f5f47f,254046,+555496663854,Nacional,Tais Braz De Ramos
4,11maehara.ingles@gmail.com,dc5c6f72-fde8-4f06-9d22-5cb5bfee2826,None,+5519983418048,Nacional,Adriano Teruo Maehara
5,13ana28abril@gmail.com,a62bed99-2bcc-4f45-a4af-2192a7c30f1c,81,+5531991327837,Nacional,Ana Maria Torres Ferreira Nunes
